# Track B, steps 1-2: text dataset and frozen Granite baseline

**Track B's question:** can the *description alone* tell us which vulnerabilities attackers will exploit? Every vulnerability has a description on the day it's published, long before EPSS scores or exploit code catch up. If text works, it helps exactly where the signals-based ranking is blind.

**What this notebook does**

1. **B1, build the text dataset.** Training uses vulnerabilities published 2018-2024, far more than the 2024-only set Track A is limited to, because text doesn't need monthly exploit signals. Validation is the same 2025 set as Phase 3, so the scores are directly comparable.
2. **B2, frozen Granite baseline.** Turn each description into 384 numbers with IBM's Granite embedding model, used as-is ("frozen"), and train a simple logistic regression on top. This is the bar the fine-tuned model (B3) must beat.

**IBM technology:** `ibm-granite/granite-embedding-small-english-r2` (47M parameters), run locally.

**Before running:** run `phase3_backtest.ipynb` first (it saves the validation set), and `pip install torch sentence-transformers duckdb pandas scikit-learn`. The embedding step takes about 50 minutes on an M3 Mac. The results are cached in `data/trackb/`, so later runs skip it.

## 1. Setup

In [1]:
import re, time, warnings
warnings.filterwarnings("ignore")  # hide library warnings (SSL, progress bars) that don't affect results
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_colwidth", 160)
DATA = next(p for p in [Path("../data"), Path("data")] if p.exists()).resolve()
OUT = DATA / "trackb"
OUT.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
MODEL = "ibm-granite/granite-embedding-small-english-r2"
print("device:", DEVICE)

con = duckdb.connect()
for t in ["vulnerabilities", "known_exploited", "scoreboard_2026"]:
    con.sql(f'''CREATE VIEW "{t}" AS
                SELECT * FROM read_csv('{DATA}/{t}.csv', quote='"', escape='"', sample_size=-1)''')

device: mps


The scoring helpers from Phase 3: `hits_at_k` counts positives in the top k (243 for validation), `bootstrap` gives a 90% range, and `compare` is the paired test (a gain is real when its range excludes 0). Rows are always sorted by `cve_id`, so results never depend on row order.

In [2]:
BENCH_K, BENCH_N = 500, 72_670
default_k = lambda n: int(round(n * BENCH_K / BENCH_N))

def rank_order(df, score, seed=0):
    cols = [score] if isinstance(score, str) else score
    if isinstance(cols, list) and all(isinstance(c, str) for c in cols):
        keys = [df[c].to_numpy(dtype=float) for c in cols]
    else:
        keys = [np.asarray(score, dtype=float)]
    keys = [np.where(np.isnan(k), -np.inf, k) for k in keys]
    tie = np.random.default_rng(seed).permutation(len(df))
    return np.lexsort([tie] + [-k for k in reversed(keys)])

def hits_at_k(df, score, k=None, seed=0):
    k = k or default_k(len(df))
    return int(df["y"].to_numpy()[rank_order(df, score, seed)[:k]].sum())

def _ranks(df, score, seed=0):
    r = np.empty(len(df), dtype=np.int64)
    r[rank_order(df, score, seed)] = np.arange(len(df))
    return r

def _boot_hits(ranks, y, k, idx):
    return np.array([y[i][np.argpartition(ranks[i], k)[:k]].sum() for i in idx])

def bootstrap(df, score, k=None, n_boot=1000, ci=0.90, seed=0):
    k = k or default_k(len(df))
    idx = np.random.default_rng(seed).integers(0, len(df), size=(n_boot, len(df)))
    h = _boot_hits(_ranks(df, score, seed), df["y"].to_numpy(), k, idx)
    return int(np.quantile(h, (1 - ci) / 2)), int(np.quantile(h, 1 - (1 - ci) / 2))

def compare(df, score_a, score_b, k=None, n_boot=1000, ci=0.90, seed=0):
    k = k or default_k(len(df))
    idx = np.random.default_rng(seed).integers(0, len(df), size=(n_boot, len(df)))
    y = df["y"].to_numpy()
    d = (_boot_hits(_ranks(df, score_a, seed), y, k, idx)
         - _boot_hits(_ranks(df, score_b, seed), y, k, idx))
    return {"diff": hits_at_k(df, score_a, k, seed) - hits_at_k(df, score_b, k, seed),
            "range": (int(np.quantile(d, (1 - ci) / 2)), int(np.quantile(d, 1 - (1 - ci) / 2))),
            "p_a_better": round(float((d > 0).mean()), 3)}

BASELINE = ["any_exploit", "epss"]

## 2. B1: the text dataset

**Training rows:** published 2018-2024, not on KEV by the end of the publish month. **Label:** `y = 1` if added to KEV after the publish month and **by December 31, 2024**. Every training label is known before 2025, so testing on 2025 stays fair.

**Validation rows:** exactly Phase 3's 2025 set (35,281 rows, 52 positives).

**Scoring rows:** the 72,670 vulnerabilities on the 2026 scoreboard, for the final handoff file.

**A caveat on older years:** KEV launched in November 2021 with a large batch of old, famous exploits. So many 2018-2021 positives are "exploited at some point" rather than "exploited within months of publication." We test two training windows, 2018-2024 and 2022-2024, and keep whichever scores better.

In [3]:
train = con.sql('''
    WITH v AS (SELECT *, CAST(published AS DATE) AS pub FROM vulnerabilities),
         k AS (SELECT cve_id, MIN(CAST(date_added AS DATE)) AS date_added FROM known_exploited GROUP BY cve_id)
    SELECT v.cve_id, v.pub AS published, v.assigner, v.description,
           CAST(k.date_added IS NOT NULL AND k.date_added <= DATE '2024-12-31' AS INT) AS y
    FROM v LEFT JOIN k USING (cve_id)
    WHERE v.pub BETWEEN DATE '2018-01-01' AND DATE '2024-12-31'
      AND (k.date_added IS NULL OR k.date_added > last_day(v.pub))
    ORDER BY v.cve_id
''').df()
val = pd.read_parquet(DATA / "backtest/val_2025.parquet").sort_values("cve_id", ignore_index=True)
score26 = con.sql('''
    SELECT s.cve_id, v.assigner, v.description FROM scoreboard_2026 s
    LEFT JOIN vulnerabilities v USING (cve_id) ORDER BY s.cve_id
''').df()

train["year"] = pd.to_datetime(train["published"]).dt.year
by_year = train.groupby("year").agg(rows=("y", "size"), positives=("y", "sum"))
by_year["rate"] = (by_year["positives"] / by_year["rows"]).map("{:.2%}".format)
print(f"train 2018-2024: {len(train):,} rows, {train.y.sum()} positives | "
      f"2022-2024: {(train.year >= 2022).sum():,} rows, {train[train.year >= 2022].y.sum()} positives")
print(f"val 2025: {len(val):,} rows, {val.y.sum()} positives | 2026 scoreboard: {len(score26):,} rows")
by_year

train 2018-2024: 165,885 rows, 684 positives | 2022-2024: 93,606 rows, 162 positives
val 2025: 35,281 rows, 52 positives | 2026 scoreboard: 72,670 rows


,rows,positives,rate
year,,,
2018,16510,71,0.43%
2019,17305,121,0.70%
2020,18322,137,0.75%
2021,20142,193,0.96%
2022,25014,66,0.26%
2023,28724,56,0.19%
2024,39868,40,0.10%


**Reading this:** 522 of the 684 positives come from 2018-2021, when KEV's launch added many old exploits at once (up to 0.96% of 2021 vulnerabilities). The rate falls to 0.26%, 0.19% and 0.10% for 2022-2024, close to the 0.15% in the 2025 validation set. The 2022-2024 window is the closer match to what we're predicting, at the cost of fewer positives.

### The input text, and one deliberate edit

The model reads `assigner | description`, e.g. `psirt@fortinet.com | A stack-based buffer overflow in FortiOS SSL-VPN allows...`. Both are Safe columns, known at publication.

**The edit: we remove sentences that say the bug was already exploited.** The team decided (Phase 4) to leave out "exploited in the wild" wording. Our descriptions are today's versions, and that sentence may have been added *after* the attack. A text model would happily learn that phrase, so we take it out of the text itself, consistently for training, validation and 2026. *Example:* "...mishandles the attaching of behavior, as exploited in the wild in February through April 2025." becomes "...mishandles the attaching of behavior."

In [4]:
IN_WILD = re.compile(r"exploited in the wild|actively exploited|may have been exploited|aware of (?:a )?report",
                     re.IGNORECASE)

def clean(description):
    # Drop whole sentences that mention exploitation; for a trailing ", as exploited..." clause, drop just the clause.
    text = re.sub(r",?\s*as (?:actively )?exploited in the wild[^.]*", "", description or "", flags=re.IGNORECASE)
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return " ".join(s for s in sentences if not IN_WILD.search(s)).strip()

for d in (train, val, score26):
    d["text"] = d["assigner"].fillna("unknown") + " | " + d["description"].fillna("").map(clean)

changed = {name: int((d["description"].fillna("").str.contains(IN_WILD)).sum()) for name, d in
           [("train", train), ("val", val), ("score26", score26)]}
print("rows with exploitation wording removed:", changed)
assert not any(d["text"].str.contains(IN_WILD).any() for d in (train, val, score26))
ex = val[val.description.str.contains("as exploited in the wild", na=False)].head(2)
for d, t in zip(ex.description, ex.text):
    print("BEFORE: ...", d[-170:], "\nAFTER:  ...", t[-170:], "\n")

rows with exploitation wording removed: {'train': 100, 'val': 23, 'score26': 13}


BEFORE: ... ocal users to terminate EDR processes and possibly have unspecified other impact via DeviceIoControl with control code 0x22E010, as exploited in the wild in October 2023. 
AFTER:  ... unwang Rentdrv2 before 2024-12-24 allows local users to terminate EDR processes and possibly have unspecified other impact via DeviceIoControl with control code 0x22E010. 

BEFORE: ... via settings update because function=savesettings lacks an authentication requirement, as exploited in the wild in May 2025. This is related to settings.php and util.php. 
AFTER:  ... ows unauthenticated command injection via settings update because function=savesettings lacks an authentication requirement. This is related to settings.php and util.php. 



**Text length:** the model reads at most 256 word pieces (tokens). This checks how many descriptions get cut off.

In [5]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)
sample = train["text"].sample(5000, random_state=0).tolist()
n_tokens = np.array([len(t) for t in tok(sample)["input_ids"]])
print(f"tokens per text: median {np.median(n_tokens):.0f}, 95th percentile {np.percentile(n_tokens, 95):.0f}, "
      f"over 256: {(n_tokens > 256).mean():.1%}")

tokens per text: median 69, 95th percentile 198, over 256: 2.8%


## 3. B2: embed every text with frozen Granite

An **embedding** is a list of numbers (here 384) that captures what a text means, so similar texts get similar numbers. *Example:* "unauthenticated RCE in a VPN gateway" and "remote attacker executes code on an SSL-VPN appliance" share few words but land close together.

We embed all ~276k texts once and save them. The cache is keyed by `cve_id`, so later notebooks (fine-tuning, Tiger Data vector search) reuse it.

In [6]:
from sentence_transformers import SentenceTransformer

def embed(df, name, batch_size=64):
    path = OUT / f"emb_{name}.npy"
    ids_path = OUT / f"emb_{name}_ids.parquet"
    if path.exists() and ids_path.exists():
        ids = pd.read_parquet(ids_path)["cve_id"]
        if ids.equals(df["cve_id"].reset_index(drop=True)):
            print(f"{name}: loaded cached embeddings")
            return np.load(path)
    model = SentenceTransformer(MODEL, device=DEVICE)
    model.max_seq_length = 256
    t0 = time.time()
    emb = model.encode(df["text"].tolist(), batch_size=batch_size, normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=False)
    np.save(path, emb.astype(np.float32))
    df[["cve_id"]].reset_index(drop=True).to_parquet(ids_path, index=False)
    print(f"{name}: embedded {len(df):,} texts in {(time.time() - t0) / 60:.1f} min")
    return emb

E = {name: embed(d, name) for name, d in [("val", val), ("train", train), ("score26", score26)]}
print({k: v.shape for k, v in E.items()})

val: loaded cached embeddings
train: loaded cached embeddings
score26: loaded cached embeddings
{'val': (35281, 384), 'train': (165885, 384), 'score26': (72670, 384)}


## 4. Train a classifier on the frozen embeddings

**Logistic regression** learns one weight per embedding number. `class_weight="balanced"` stops the rare positives (0.4% of training rows) from being drowned out. The settings are fixed in advance (`C=1.0`), not tuned on 2025, so the validation score stays honest.

We train on both windows and score the 2025 validation set.

In [7]:
def fit_text_model(mask):
    model = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, class_weight="balanced", max_iter=3000))
    model.fit(E["train"][mask], train["y"][mask])
    return model

results, text_scores = [], {}
for label, mask in [("2018-2024", np.ones(len(train), bool)), ("2022-2024", (train.year >= 2022).to_numpy())]:
    t0 = time.time()
    model = fit_text_model(mask)
    s = model.predict_proba(E["val"])[:, 1]
    text_scores[label] = s
    c = compare(val, s, BASELINE)
    results.append({"text model trained on": label, "train positives": int(train.y[mask].sum()),
                    "hits@243": hits_at_k(val, s), "90% range": "%d-%d" % bootstrap(val, s),
                    "vs. baseline (14)": c["diff"], "diff range": "%+d to %+d" % c["range"],
                    "fit time (s)": round(time.time() - t0)})
results.insert(0, {"text model trained on": "Baseline: exploit code then EPSS", "hits@243": hits_at_k(val, BASELINE),
                   "90% range": "%d-%d" % bootstrap(val, BASELINE)})
pd.DataFrame(results).fillna("")

,text model trained on,hits@243,90% range,train positives,vs. baseline (14),diff range,fit time (s)
0,Baseline: exploit code then EPSS,14,8-21,,,,
1,2018-2024,4,1-8,684.0,-10.0,-16 to -5,4.0
2,2022-2024,4,1-8,162.0,-10.0,-18 to -4,3.0


**Reading this: frozen text alone is weak at the top.** Both training windows find 4 of 52 in the top 243 (range 1-8), 10 fewer than the baseline, and the paired range (about -16 to -4) says that gap is real. The older 2018-2021 positives neither help nor hurt. Both windows score the same, so we keep the larger 2018-2024 set.

### Is there signal deeper in the ranking?

hits@243 only looks at the very top. Two wider views:
- **hits at larger cut-offs:** 243, 500, 1,000, 2,000 and 5,000
- **AUC:** the chance that a random exploited vulnerability is ranked above a random non-exploited one. 0.5 is a coin flip and 1.0 is perfect.

In [8]:
from sklearn.metrics import roc_auc_score
# Both windows tie at the top, so max() keeps the first: the larger 2018-2024 set.
best_window = max(text_scores, key=lambda w: hits_at_k(val, text_scores[w]))
s_text = text_scores[best_window]
rows = []
for k in [243, 500, 1000, 2000, 5000]:
    rows.append({"top k": k, "text (frozen)": hits_at_k(val, s_text, k), "EPSS first": hits_at_k(val, "epss", k),
                 "exploit code then EPSS": hits_at_k(val, BASELINE, k)})
print(f"AUC: text {roc_auc_score(val.y, s_text):.3f} | EPSS {roc_auc_score(val.y, val.epss.fillna(0)):.3f}")
pd.DataFrame(rows)

AUC: text 0.800 | EPSS 0.707


,top k,text (frozen),EPSS first,exploit code then EPSS
0,243,4,8,14
1,500,6,8,18
2,1000,9,13,22
3,2000,19,18,24
4,5000,26,28,33


**Reading this: the text knows something, but it isn't sharp.** Its AUC (0.80) is *higher* than EPSS's (0.71), so overall it separates exploited from non-exploited vulnerabilities better. By the top 2,000 it has caught up with EPSS (19 vs. 18). What it lacks is precision in the first few hundred slots, which is where we're scored.

That's what we'd expect from a *frozen* model. Granite's general-purpose embedding knows "this is a remote code execution in a network appliance," but it was never taught which of those details matter for exploitation. Fine-tuning (B3) trains exactly that.

### Text plus signals: a rank-average blend

The text model is meant to *add* to the signals ranking, not replace it. A **rank average** turns each score into a rank (1 = most dangerous) and averages the ranks, so neither score's scale matters. *Example:* a VPN bug ranked 900th by the baseline (no EPSS yet) but 50th by text averages to 475th.

The weights are fixed in advance, as the team plan suggests: 50/50 and 70/30 (baseline/text).

In [9]:
def rank_pct(df, score):
    # 0 = best, 1 = worst
    return _ranks(df, score) / (len(df) - 1)

blends = {}
for w in [0.5, 0.7]:
    blends[w] = -(w * rank_pct(val, BASELINE) + (1 - w) * rank_pct(val, s_text))  # higher = better
rows = []
for w, s in blends.items():
    c = compare(val, s, BASELINE)
    rows.append({"blend (baseline/text)": f"{w:.0%} / {1 - w:.0%}", "hits@243": hits_at_k(val, s),
                 "90% range": "%d-%d" % bootstrap(val, s), "vs. baseline": c["diff"],
                 "diff range": "%+d to %+d" % c["range"], "P(better)": c["p_a_better"]})
print("text model used:", best_window)
pd.DataFrame(rows)

text model used: 2018-2024


,blend (baseline/text),hits@243,90% range,vs. baseline,diff range,P(better)
0,50% / 50%,11,6-17,-3,-9 to +2,0.124
1,70% / 30%,14,8-19,0,-7 to +4,0.305


**Reading this:** the 70/30 blend ties the baseline at 14 (paired range -7 to +4), and 50/50 drops to 11. With frozen text this weak at the top, giving it more weight pulls good baseline picks out of the top 243. The blend becomes worth revisiting once the fine-tuned score is sharper.

### Where does text find what EPSS misses?

The positives the text model puts in its top 243 that the EPSS-first ranking places below 5,000th. Those are "what the description knew before EPSS did," and they're the Discovery angle for Track B.

In [10]:
ranked = val.assign(text_rank=_ranks(val, s_text), epss_rank=_ranks(val, "epss"))
pos = ranked[ranked.y == 1]
for top in [243, 2000]:
    n = ((pos.text_rank < top) & (pos.epss_rank >= 5000)).sum()
    print(f"positives in text's top {top:,} that EPSS ranks 5,000th or lower: {n}")
pos[(pos.text_rank < 2000) & (pos.epss_rank >= 5000)][["cve_id", "text_rank", "epss_rank", "text"]].sort_values("text_rank")

positives in text's top 243 that EPSS ranks 5,000th or lower: 0
positives in text's top 2,000 that EPSS ranks 5,000th or lower: 6


,cve_id,text_rank,epss_rank,text
26105,CVE-2025-47827,465,31683,"cve@mitre.org | In IGEL OS before 11, Secure Boot can be bypassed because the igel-flash-driver module improperly verifies a cryptographic signature. Ultima..."
32530,CVE-2025-6554,677,33253,chrome-cve-admin@google.com | Type confusion in V8 in Google Chrome prior to 138.0.7204.96 allowed a remote attacker to perform arbitrary read/write via a c...
7441,CVE-2024-57968,852,10281,"cve@mitre.org | Advantive VeraCore before 2024.4.2.1 allows remote authenticated users to upload files to unintended folders (e.g., ones that are accessible..."
14330,CVE-2025-24472,1468,13236,psirt@fortinet.com | An Authentication Bypass Using an Alternate Path or Channel vulnerability [CWE-288] affecting FortiOS 7.0.0 through 7.0.16 and FortiPro...
2209,CVE-2023-52163,1517,34707,cve@mitre.org | Digiever DS-2105 Pro 3.1.0.71-11 devices allow time_tzsetup.cgi Command Injection. NOTE: This vulnerability only affects products that are n...
30867,CVE-2025-5777,1952,6669,"secure@citrix.com | Insufficient input validation leading to memory overread when the NetScaler is configured as a Gateway (VPN virtual server, ICA Proxy, C..."


**Reading this:** none of the 52 positives are in the text's top 243 while EPSS misses them. But 6 are in the text's top 2,000 while EPSS ranks them 5,000th or lower. Those are vulnerabilities whose description looked dangerous before EPSS noticed. If fine-tuning lifts some of them into the top few hundred, that's Track B's Discovery story.

## 5. Save the frozen-baseline scores

These go into the Compare and combine table, and they're the fallback text score if fine-tuning (B3) doesn't beat them. The format is the team's handoff format: `cve_id, text_score`.

In [11]:
pd.DataFrame({"cve_id": val["cve_id"], "text_score": s_text}).to_csv(OUT / "text_val_2025_frozen.csv", index=False)
print("Saved:", *sorted(p.name for p in OUT.iterdir()))

Saved: emb_score26.npy emb_score26_ids.parquet emb_train.npy emb_train_ids.parquet emb_val.npy emb_val_ids.parquet text_val_2025_frozen.csv


## Summary

- **B1 dataset:** 165,885 training descriptions (2018-2024, 684 positives), the 2025 validation set (35,281, 52 positives) and the 2026 scoreboard (72,670). "Exploited in the wild" wording is removed everywhere: 100 training rows had it, and 35 of them were positives, so leaving it in would have leaked.
- **Embeddings:** all 273,836 texts, cached in `data/trackb/` for fine-tuning and the Tiger Data vector search.
- **Frozen Granite + logistic regression:** 4 hits@243, far below the baseline's 14. But its AUC (0.80) beats EPSS (0.71), and it matches EPSS in the top 2,000. **The signal is there; it isn't sharp enough at the top yet.**
- **Blend with the baseline:** 14 at 70/30, so no gain yet.

**Next (B3):** fine-tune Granite end to end as a classifier, so the model learns which details of a description matter for exploitation. The bar to beat is 4 (frozen) for the text score on its own, and 14 for the blend.